<a href="https://colab.research.google.com/github/romanakki23/Flyrank_my_work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romanakki23/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row represents a single pseudonymized content item (content_hash_id) for a specific client (client_hash_id) on a specific report date (report_date).

Time Window: Features are aggregated over a historical 90-day feature window ending on 2026-03-31 (mid-panel month month=2026-03), predicting a binary target flag (is_declining) evaluated in the subsequent performance period.

In [10]:
# Section 1 verification code
import os
import pandas as pd

# Load starter data to confirm local grain
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), None)
if not csv_path:
    csv_path = "https://raw.githubusercontent.com/romanakki23/flyrank/main/data/raw/content_refresh_anonymized.csv"

df_starter = pd.read_csv(csv_path)
print(f"Starter Dataset Grain Check: {len(df_starter):,} total content pages.")

Starter Dataset Grain Check: 30,000 total content pages.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions_90d, clicks_90d, sessions_90d, avg_position, content_age_days, ctr, engagement_rate, scroll_rate. (Knowable prior to prediction point).

Label / Proxy: is_declining_label (trend_direction == 'down').

Context: content_type, main_intent, competition_level, word_count_tier.

Excluded: health_score, priority_score, action_type, raw client names, raw URLs, raw query strings.

Reason: Product-derived rule scores are excluded to prevent circular results; raw identifiers are scrambled/excluded for privacy safety.

In [11]:
# Bucket verification
print(
    f"Features loaded: {len([c for c in df_starter.columns if '_90d' in c])} signal columns."
)

Features loaded: 8 signal columns.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Below we execute three verification checks (grain uniqueness, row counts/spans, and availability flags) plus the deliberate feature leakage experiment.

In [8]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Load starter dataset
possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), None)
if not csv_path:
    csv_path = "https://raw.githubusercontent.com/romanakki23/flyrank/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)

# 2. Filter minimum criteria & create target
df_sample = df[(df["impressions_90d"] >= 100) & (df["content_age_days"] >= 90)].copy()
df_sample["is_declining_label"] = (df_sample["trend_direction"] == "down").astype(int)

# 3. Define 5 knowable features
features = [
    "impressions_90d",
    "sessions_90d",
    "avg_position",
    "content_age_days",
    "ctr",
]

# Create deliberate leaked feature
df_sample["leaked_future_feature"] = (df_sample["is_declining_label"] * 0.99) + 0.01

# Train/Test Split to prevent artificial 1.00 overfitting
X_train, X_test, y_train, y_test = train_test_split(
    df_sample[features + ["leaked_future_feature"]],
    df_sample["is_declining_label"],
    test_size=0.3,
    random_state=42,
)

# --- HONEST MODEL ---
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
clf_honest.fit(X_train[features], y_train)
score_honest = roc_auc_score(y_test, clf_honest.predict_proba(X_test[features])[:, 1])

# --- LEAKED MODEL ---
clf_leaked = RandomForestClassifier(n_estimators=100, random_state=42)
clf_leaked.fit(X_train[features + ["leaked_future_feature"]], y_train)
score_leaked = roc_auc_score(
    y_test, clf_leaked.predict_proba(X_test[features + ["leaked_future_feature"]])[:, 1]
)

print("--- LEAKAGE TRAP RESULTS (Out-of-Sample) ---")
print(f"ROC AUC WITH Leaked Column (Artificial Score): {score_leaked:.4f}")
print(f"ROC AUC WITHOUT Leaked Column (Honest Score):    {score_honest:.4f}")

--- LEAKAGE TRAP RESULTS (Out-of-Sample) ---
ROC AUC WITH Leaked Column (Artificial Score): 1.0000
ROC AUC WITHOUT Leaked Column (Honest Score):    0.7085


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced Client Tracking History: Different clients have varying tracking start dates (gsc_data_start, ga4_data_start). Treating missing GA4 metrics as zero traffic without filtering on ga4_data_available IS TRUE introduces artificial decline signals.

Observational Nature: This dataset cannot prove causal recovery (e.g., editing a page guarantees a recovery). It serves as decision-support to optimize review efficiency.

In [9]:
# Confirm GA4 availability distribution on local starter slice
ga4_coverage = (df_starter["sessions_90d"] > 0).mean() * 100
print(f"Local starter slice GA4 activity density: {ga4_coverage:.1f}%")

Local starter slice GA4 activity density: 100.0%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.